# 04 — Naive and Moving Average Forecast

**Gold Nexus Alpha — professor-style forecasting pipeline**

Purpose of this notebook:

1. Load the long univariate gold dataset created by Notebook 03.
2. Convert `gold_price` into a time-series object using `date` as the index.
3. Plot the time series before modeling, matching the professor's Amtrak workflow.
4. Partition the series by locked train / validation / test windows.
5. Build a Naive one-step baseline.
6. Build trailing moving-average baselines for 5, 10, 20, and 60 weekday windows.
7. Calculate MAE, RMSE, MAPE, bias / mean error, and directional accuracy.
8. Export JSON artifacts for the Vercel frontend.
9. Push outputs to GitHub.

Professor reference mirrored from the uploaded Amtrak notebooks:

- `pd.to_datetime(...)`
- `pd.Series(values, index=date, name=...)`
- time-series plot before modeling
- time-based partitioning
- Naive forecast
- trailing moving average
- shift by one period to avoid using the current target value
- residual/error plots
- MSE / MAE / MAPE-style accuracy reporting

Professor-safe rule: every forecast used for backtesting is **one-step roll-forward**. For date `t`, the prediction only uses information available before date `t`.


In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the same clean Colab → GitHub artifact workflow as Notebooks 01–03.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset: remove old clone if present.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

# Clone URL. Use token for private/push access, but never print the token.
public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

# Git identity for Colab commits.
run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))

# Ensure push remote uses token when available.
if GITHUB_TOKEN:
    run_cmd(
        ["git", "remote", "set-url", "origin", auth_url],
        cwd=str(PROJECT_DIR),
        display_cmd=["git", "remote", "set-url", "origin", f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git"]
    )

print("✅ Repository ready:", PROJECT_DIR)


In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load the same core time-series libraries used in the professor's Amtrak notebooks.
# - Avoid heavy ML libraries for this baseline notebook.
# ======================================================================================

import json
import math
import glob
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

%matplotlib inline

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Notebook 03 output: data/aligned/model_ready_univariate.csv
# - Fall back to weekday_clean_matrix.csv or an uploaded Gold_Matrix CSV only if needed.
# - Lock the professor-safe univariate dataset and time splits.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Locked project rules.
OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

LONG_UNIVARIATE_START = "1968-01-04"
LONG_UNIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "1968-01-04"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

MA_WINDOWS = [5, 10, 20, 60]

def find_input_file():
    """Find the most appropriate input file for Notebook 04."""
    candidates = [
        ALIGNED_DIR / "model_ready_univariate.csv",
        ALIGNED_DIR / "weekday_clean_matrix.csv",
    ]

    # Also allow Colab-uploaded files as fallback.
    upload_patterns = [
        "/content/model_ready_univariate*.csv",
        "/content/weekday_clean_matrix*.csv",
        "/content/Gold_Matrix*.csv",
    ]
    for pattern in upload_patterns:
        candidates.extend(Path(p) for p in glob.glob(pattern))

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find model_ready_univariate.csv, weekday_clean_matrix.csv, or uploaded Gold_Matrix CSV. "
        "Run Notebooks 01–03 first or upload the matrix CSV."
    )

INPUT_PATH = find_input_file()
print("✅ Input file detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
print("Raw input shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")
if "gold_price" not in raw_df.columns:
    raise ValueError("Input file must contain a 'gold_price' column.")

# Standardize date and sort.
df = raw_df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last")

# If fallback is raw matrix, enforce weekday cleaning here.
df["weekday"] = df["date"].dt.dayofweek
if df["weekday"].isin([5, 6]).any():
    before = len(df)
    df = df[~df["weekday"].isin([5, 6])].copy()
    after = len(df)
    print(f"⚠️ Fallback cleaning applied: removed {before - after:,} Saturday/Sunday rows.")

# Lock official univariate window and target.
df = df[(df["date"] >= pd.Timestamp(LONG_UNIVARIATE_START)) & (df["date"] <= pd.Timestamp(LONG_UNIVARIATE_END))].copy()
df["gold_price"] = pd.to_numeric(df["gold_price"], errors="coerce")
df = df.dropna(subset=["gold_price"]).copy()

# Assign locked split labels if not already present.
def assign_split(d):
    d = pd.Timestamp(d)
    if pd.Timestamp(TRAIN_START) <= d <= pd.Timestamp(TRAIN_END):
        return "train"
    if pd.Timestamp(VALIDATION_START) <= d <= pd.Timestamp(VALIDATION_END):
        return "validation"
    if pd.Timestamp(TEST_START) <= d <= pd.Timestamp(TEST_END):
        return "test"
    return "outside_model_window"

df["split"] = df["date"].apply(assign_split)

model_df = df[df["split"].isin(["train", "validation", "test"])][["date", "gold_price", "split"]].copy()
model_df = model_df.sort_values("date").reset_index(drop=True)

print("✅ Model dataframe shape:", model_df.shape)
print(model_df.groupby("split").agg(rows=("date", "count"), start=("date", "min"), end=("date", "max")))

display(model_df.head())
display(model_df.tail())


In [ ]:
# ======================================================================================
# CELL 4 — MAIN NAIVE AND MOVING AVERAGE MODELING LOGIC
# ======================================================================================
# Purpose:
# - Mirror professor Amtrak flow: series creation → plot → partition → forecast → residuals → accuracy.
# - Use leakage-safe one-step roll-forward predictions for validation and test metrics.
# ======================================================================================

# --------------------------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------------------------

def to_float_or_none(x):
    if x is None:
        return None
    try:
        if pd.isna(x) or np.isinf(x):
            return None
        return float(x)
    except Exception:
        return None

def safe_mape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = (~np.isnan(actual)) & (~np.isnan(pred)) & (actual != 0)
    if mask.sum() == 0:
        return None
    return float(np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100)

def directional_accuracy(actual_series, pred_series):
    """
    Directional accuracy compares the sign of actual change vs predicted change.
    This is optional, but useful for finance interpretation.
    """
    temp = pd.DataFrame({"actual": actual_series, "pred": pred_series}).dropna()
    if len(temp) < 3:
        return None
    actual_direction = np.sign(temp["actual"].diff())
    pred_direction = np.sign(temp["pred"].diff())
    valid = actual_direction.notna() & pred_direction.notna()
    if valid.sum() == 0:
        return None
    return float((actual_direction[valid] == pred_direction[valid]).mean() * 100)

def calculate_metrics(frame, actual_col, pred_col):
    temp = frame[[actual_col, pred_col]].dropna().copy()
    if temp.empty:
        return {
            "n": 0,
            "MAE": None,
            "MSE": None,
            "RMSE": None,
            "MAPE": None,
            "mean_error_bias": None,
            "directional_accuracy_pct": None,
        }

    errors = temp[actual_col] - temp[pred_col]
    mae = np.mean(np.abs(errors))
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mape = safe_mape(temp[actual_col], temp[pred_col])
    bias = np.mean(errors)

    return {
        "n": int(len(temp)),
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": float(rmse),
        "MAPE": mape,
        "mean_error_bias": float(bias),
        "directional_accuracy_pct": directional_accuracy(temp[actual_col], temp[pred_col]),
    }

def metrics_by_split(frame, pred_col):
    return {
        split_name: calculate_metrics(frame[frame["split"] == split_name], "gold_price", pred_col)
        for split_name in ["train", "validation", "test"]
    }

def professor_static_holdout_metrics(ts, train_index, validation_index, test_index, window=None):
    """
    Mirrors the professor's Amtrak holdout idea:
    - Validation forecast is based on the last training value / last training moving average.
    - Test forecast is based on the last train+validation value / last train+validation moving average.
    This is included as a class-style illustration. Primary backtest metrics below use one-step roll-forward.
    """
    train_ts = ts.loc[train_index]
    validation_ts = ts.loc[validation_index]
    test_ts = ts.loc[test_index]
    train_validation_ts = ts.loc[train_index.union(validation_index)]

    if window is None:
        valid_pred = pd.Series(train_ts.iloc[-1], index=validation_ts.index)
        test_pred = pd.Series(train_validation_ts.iloc[-1], index=test_ts.index)
        label = "naive_static_holdout"
    else:
        valid_level = train_ts.rolling(window).mean().iloc[-1]
        test_level = train_validation_ts.rolling(window).mean().iloc[-1]
        valid_pred = pd.Series(valid_level, index=validation_ts.index)
        test_pred = pd.Series(test_level, index=test_ts.index)
        label = f"ma_{window}_static_holdout"

    valid_frame = pd.DataFrame({"gold_price": validation_ts, label: valid_pred})
    test_frame = pd.DataFrame({"gold_price": test_ts, label: test_pred})

    return {
        "validation": calculate_metrics(valid_frame, "gold_price", label),
        "test": calculate_metrics(test_frame, "gold_price", label),
    }

# --------------------------------------------------------------------------------------
# Professor-style time series setup
# --------------------------------------------------------------------------------------

gold_ts = pd.Series(
    model_df["gold_price"].values,
    index=pd.DatetimeIndex(model_df["date"]),
    name="gold_price"
)

# Weekday-clean data does not remove market holidays yet, so frequency may not infer perfectly.
try:
    inferred_freq = pd.infer_freq(gold_ts.index)
except Exception:
    inferred_freq = None

print("Gold time series created")
print("Start:", gold_ts.index.min().date())
print("End:", gold_ts.index.max().date())
print("Observations:", len(gold_ts))
print("Inferred frequency:", inferred_freq)

# Plot the full series, matching professor's first visualization step.
ax = gold_ts.plot(figsize=(12, 5), linewidth=0.8)
ax.set_xlabel("Time")
ax.set_ylabel("Gold Price")
ax.set_title("Gold Price Time Series")
plt.show()

# Partition the data.
train_df = model_df[model_df["split"] == "train"].copy()
validation_df = model_df[model_df["split"] == "validation"].copy()
test_df = model_df[model_df["split"] == "test"].copy()

train_index = pd.DatetimeIndex(train_df["date"])
validation_index = pd.DatetimeIndex(validation_df["date"])
test_index = pd.DatetimeIndex(test_df["date"])

print("Partition sizes")
print("Train:", len(train_df), train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Validation:", len(validation_df), validation_df["date"].min().date(), "to", validation_df["date"].max().date())
print("Test:", len(test_df), test_df["date"].min().date(), "to", test_df["date"].max().date())

# --------------------------------------------------------------------------------------
# Naive Method
# --------------------------------------------------------------------------------------
# Professor logic: naive forecast uses the previous observed value.
# For one-step roll-forward evaluation, prediction at date t = gold_price at t-1.

results_df = model_df.copy()
results_df["naive_pred"] = results_df["gold_price"].shift(1)

naive_metrics = metrics_by_split(results_df, "naive_pred")
naive_static_metrics = professor_static_holdout_metrics(
    gold_ts,
    train_index=train_index,
    validation_index=validation_index,
    test_index=test_index,
    window=None,
)

print("Naive one-step roll-forward metrics")
display(pd.DataFrame(naive_metrics).T)

# Naive plot: focus on validation + test with a little training context.
plot_start = pd.Timestamp("2018-01-01")
plot_df = results_df[results_df["date"] >= plot_start].copy()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(plot_df["date"], plot_df["gold_price"], label="Actual", linewidth=1.2)
ax.plot(plot_df["date"], plot_df["naive_pred"], label="Naive Forecast", linestyle="--", linewidth=1.0)
ax.axvline(pd.Timestamp(VALIDATION_START), linewidth=1)
ax.axvline(pd.Timestamp(TEST_START), linewidth=1)
ax.set_xlabel("Time")
ax.set_ylabel("Gold Price")
ax.set_title("Actual vs Naive Forecast — One-Step Roll-Forward")
ax.legend()
plt.show()

# Naive residual plot for validation/test.
eval_naive = results_df[results_df["split"].isin(["validation", "test"])].copy()
eval_naive["naive_residual"] = eval_naive["gold_price"] - eval_naive["naive_pred"]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(eval_naive["date"], eval_naive["naive_residual"], label="Naive Residual", linewidth=0.8)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Time")
ax.set_ylabel("Actual - Forecast")
ax.set_title("Naive Forecast Residuals")
ax.legend()
plt.show()

# --------------------------------------------------------------------------------------
# Moving Average Method
# --------------------------------------------------------------------------------------
# Professor logic: trailing moving average, shifted by one period to avoid using current actual.
# Prediction at date t = mean of prior N observed gold prices.

ma_metric_rows = []
moving_average_metrics = {}
moving_average_static_metrics = {}

for window in MA_WINDOWS:
    pred_col = f"ma_{window}_pred"
    results_df[pred_col] = results_df["gold_price"].shift(1).rolling(window=window).mean()

    split_metrics = metrics_by_split(results_df, pred_col)
    moving_average_metrics[str(window)] = split_metrics

    moving_average_static_metrics[str(window)] = professor_static_holdout_metrics(
        gold_ts,
        train_index=train_index,
        validation_index=validation_index,
        test_index=test_index,
        window=window,
    )

    for split_name, metric_values in split_metrics.items():
        ma_metric_rows.append({
            "window": window,
            "split": split_name,
            **metric_values,
        })

ma_metrics_df = pd.DataFrame(ma_metric_rows)
print("Moving Average one-step roll-forward metrics")
display(ma_metrics_df)

# Choose best MA window by validation RMSE only. This is not the final project winner.
valid_ma_metrics = ma_metrics_df[ma_metrics_df["split"] == "validation"].copy()
best_ma_row = valid_ma_metrics.sort_values("RMSE", ascending=True).iloc[0].to_dict()
best_ma_window = int(best_ma_row["window"])
best_ma_col = f"ma_{best_ma_window}_pred"

print(f"Best moving-average baseline by validation RMSE: {best_ma_window}-day trailing MA")
display(pd.DataFrame([best_ma_row]))

# Plot actual vs all moving-average forecasts for validation/test context.
plot_df = results_df[results_df["date"] >= plot_start].copy()

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(plot_df["date"], plot_df["gold_price"], label="Actual", linewidth=1.3)
for window in MA_WINDOWS:
    ax.plot(plot_df["date"], plot_df[f"ma_{window}_pred"], label=f"MA {window}", linestyle="--", linewidth=0.9)
ax.axvline(pd.Timestamp(VALIDATION_START), linewidth=1)
ax.axvline(pd.Timestamp(TEST_START), linewidth=1)
ax.set_xlabel("Time")
ax.set_ylabel("Gold Price")
ax.set_title("Actual vs Trailing Moving Average Forecasts")
ax.legend()
plt.show()

# Residual plot for best MA window.
eval_ma = results_df[results_df["split"].isin(["validation", "test"])].copy()
eval_ma["best_ma_residual"] = eval_ma["gold_price"] - eval_ma[best_ma_col]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(eval_ma["date"], eval_ma["best_ma_residual"], label=f"MA {best_ma_window} Residual", linewidth=0.8)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Time")
ax.set_ylabel("Actual - Forecast")
ax.set_title(f"Best Moving Average Residuals — {best_ma_window}-Day Window")
ax.legend()
plt.show()

# Combined baseline leaderboard for validation/test.
leaderboard_rows = []
leaderboard_rows.append({"model": "Naive", "window": None, "split": "validation", **naive_metrics["validation"]})
leaderboard_rows.append({"model": "Naive", "window": None, "split": "test", **naive_metrics["test"]})
for _, row in ma_metrics_df[ma_metrics_df["split"].isin(["validation", "test"])].iterrows():
    leaderboard_rows.append({
        "model": f"Moving Average {int(row['window'])}",
        "window": int(row["window"]),
        "split": row["split"],
        "n": int(row["n"]),
        "MAE": row["MAE"],
        "MSE": row["MSE"],
        "RMSE": row["RMSE"],
        "MAPE": row["MAPE"],
        "mean_error_bias": row["mean_error_bias"],
        "directional_accuracy_pct": row["directional_accuracy_pct"],
    })

baseline_leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values(["split", "RMSE"]).reset_index(drop=True)
print("Baseline leaderboard")
display(baseline_leaderboard_df)


In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Notebook 04 JSON artifacts for model pages and future validation notebook.
# ======================================================================================

def json_safe(obj):
    """Convert pandas/numpy objects into JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, (np.ndarray,)):
        return json_safe(obj.tolist())
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, np.ndarray)) else False:
        return None
    return obj

def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, ensure_ascii=False)
    print("✅ Wrote JSON:", path.relative_to(PROJECT_DIR))

def records_from_df(frame: pd.DataFrame):
    export = frame.copy()
    for col in export.columns:
        if np.issubdtype(export[col].dtype, np.datetime64):
            export[col] = export[col].dt.strftime("%Y-%m-%d")
    return json_safe(export.replace({np.nan: None}).to_dict(orient="records"))

generated_at = datetime.now(timezone.utc).isoformat()

# Keep chart path compact enough for frontend: final training context + validation + test.
chart_start = pd.Timestamp("2018-01-01")
forecast_path_cols = ["date", "split", "gold_price", "naive_pred"] + [f"ma_{w}_pred" for w in MA_WINDOWS]
chart_df = results_df[results_df["date"] >= chart_start][forecast_path_cols].copy()

# Full evaluation data: validation and test only.
eval_path_df = results_df[results_df["split"].isin(["validation", "test"])][forecast_path_cols].copy()

naive_results = {
    "artifact_name": "naive_results",
    "notebook": "04_naive_and_moving_average_forecast.ipynb",
    "generated_at_utc": generated_at,
    "model": {
        "name": "Naive Forecast",
        "category": "baseline",
        "description": "One-step naive baseline where the forecast for date t equals the previous observed gold price.",
    },
    "dataset": {
        "source_file": str(INPUT_PATH.name),
        "target": "gold_price",
        "frequency_note": "Weekday-clean daily series; market holidays have not been removed.",
        "window": {
            "start": LONG_UNIVARIATE_START,
            "end": LONG_UNIVARIATE_END,
        },
        "splits": {
            "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": int((results_df["split"] == "train").sum())},
            "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": int((results_df["split"] == "validation").sum())},
            "test": {"start": TEST_START, "end": TEST_END, "rows": int((results_df["split"] == "test").sum())},
        },
    },
    "evaluation_design": {
        "primary_method": "one_step_roll_forward",
        "primary_method_note": "For each date t, the forecast uses only gold prices available before date t.",
        "professor_static_holdout_included": True,
        "professor_static_holdout_note": "Included because the professor's Amtrak class example also shows validation forecasts created from the last training observation.",
    },
    "metrics": naive_metrics,
    "professor_static_holdout_metrics": naive_static_metrics,
    "limitations": [
        "The naive model does not use trend, seasonality, macro factors, or structural gold drivers.",
        "It is a baseline benchmark only, not a final forecasting claim.",
        "Strong performance by naive models in financial series can reflect persistence rather than explanatory power.",
    ],
}

moving_average_results = {
    "artifact_name": "moving_average_results",
    "notebook": "04_naive_and_moving_average_forecast.ipynb",
    "generated_at_utc": generated_at,
    "model": {
        "name": "Trailing Moving Average Forecast",
        "category": "baseline",
        "windows_tested": MA_WINDOWS,
        "description": "Trailing moving-average baselines where the forecast for date t is the average of prior observed gold prices.",
    },
    "dataset": naive_results["dataset"],
    "evaluation_design": naive_results["evaluation_design"],
    "metrics_by_window": moving_average_metrics,
    "metrics_table": records_from_df(ma_metrics_df),
    "professor_static_holdout_metrics_by_window": moving_average_static_metrics,
    "best_moving_average_by_validation_rmse": {
        "window": best_ma_window,
        "selection_basis": "Lowest validation RMSE among moving-average windows only.",
        "warning": "This is not the final project winner. Final selection happens only in Notebook 11 after all models are compared.",
        "metrics": json_safe(best_ma_row),
    },
    "limitations": [
        "Moving averages smooth short-term noise but lag during fast trend changes.",
        "Window selection is sensitive to the validation period.",
        "Moving averages do not directly use macro or financial explanatory variables.",
        "This notebook does not claim any final winning model.",
    ],
}

baseline_forecast_paths = {
    "artifact_name": "baseline_forecast_paths",
    "notebook": "04_naive_and_moving_average_forecast.ipynb",
    "generated_at_utc": generated_at,
    "chart_data_start": chart_start.strftime("%Y-%m-%d"),
    "chart_data_note": "Includes final training context plus validation and test periods for frontend charts.",
    "columns": forecast_path_cols,
    "chart_data": records_from_df(chart_df),
    "evaluation_path_data": records_from_df(eval_path_df),
}

page_naive_moving_average = {
    "artifact_name": "page_naive_moving_average",
    "page_title": "Naive + Moving Average Forecasts",
    "page_subtitle": "Professor-style baseline time-series forecasts for gold price.",
    "generated_at_utc": generated_at,
    "source_artifacts": [
        "artifacts/models/naive_results.json",
        "artifacts/models/moving_average_results.json",
        "artifacts/models/baseline_forecast_paths.json",
    ],
    "what_this_page_shows": [
        "A naive benchmark where tomorrow's forecast equals the prior observed gold price.",
        "Trailing moving-average benchmarks using 5, 10, 20, and 60 weekday windows.",
        "Validation and test performance using leakage-safe one-step roll-forward predictions.",
        "Residual charts and baseline metrics for later model comparison.",
    ],
    "dataset_window": {
        "start": LONG_UNIVARIATE_START,
        "end": LONG_UNIVARIATE_END,
        "target": "gold_price",
        "cutoff": OFFICIAL_FORECAST_CUTOFF_DATE,
    },
    "splits": naive_results["dataset"]["splits"],
    "models": [
        {
            "name": "Naive Forecast",
            "type": "baseline",
            "metrics": naive_metrics,
        },
        {
            "name": "Moving Average",
            "type": "baseline",
            "windows_tested": MA_WINDOWS,
            "best_window_by_validation_rmse": best_ma_window,
            "metrics_table": records_from_df(ma_metrics_df),
        },
    ],
    "leaderboard": records_from_df(baseline_leaderboard_df),
    "primary_chart": {
        "artifact": "artifacts/models/baseline_forecast_paths.json",
        "x": "date",
        "y_actual": "gold_price",
        "y_forecasts": ["naive_pred"] + [f"ma_{w}_pred" for w in MA_WINDOWS],
    },
    "kpi_cards": [
        {
            "label": "Naive Validation RMSE",
            "value": naive_metrics["validation"]["RMSE"],
            "unit": "gold price units",
        },
        {
            "label": "Naive Test RMSE",
            "value": naive_metrics["test"]["RMSE"],
            "unit": "gold price units",
        },
        {
            "label": "Best MA Window",
            "value": best_ma_window,
            "unit": "weekdays",
        },
        {
            "label": "Best MA Validation RMSE",
            "value": best_ma_row["RMSE"],
            "unit": "gold price units",
        },
    ],
    "limitations": [
        "These are baseline models only.",
        "They are included to create a fair benchmark before advanced models.",
        "The final selected model cannot be decided until Notebook 11 model comparison.",
    ],
}

# Export required artifacts.
write_json(MODELS_ARTIFACTS_DIR / "naive_results.json", naive_results)
write_json(MODELS_ARTIFACTS_DIR / "moving_average_results.json", moving_average_results)
write_json(MODELS_ARTIFACTS_DIR / "baseline_forecast_paths.json", baseline_forecast_paths)
write_json(PAGES_ARTIFACTS_DIR / "page_naive_moving_average.json", page_naive_moving_average)

print("✅ Notebook 04 artifact export complete.")


In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 04 outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/naive_results.json",
    "artifacts/models/moving_average_results.json",
    "artifacts/models/baseline_forecast_paths.json",
    "artifacts/pages/page_naive_moving_average.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Add Notebook 04 naive and moving average forecast artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ Notebook 04 artifacts pushed to GitHub.")
else:
    print("ℹ️ No changes detected. Nothing to commit.")

print("✅ Notebook 04 complete.")
